In [1]:
# ############################################################
# Level 2 — 유방암 분류 + 스케일링 + ROC-AUC / 상세 성적표
# ############################################################
# ------------------------------------------------------------
# [목적] 도구 불러오기 — 데이터 / 분리 / 전처리 / 모델 / 평가
# ------------------------------------------------------------
from sklearn.datasets import load_breast_cancer   # 유방암 데이터
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler  # 스케일링 (특성 크기 맞추기)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report  # AUC + 상세 성적표

In [2]:
# ------------------------------------------------------------
# [목적] 데이터 준비 + 8:2 분리
# ------------------------------------------------------------
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)   # 비율 유지 8:2

In [3]:
# ------------------------------------------------------------
# [목적] 스케일링 — train으로 기준을 배우고 test엔 그 기준만 적용 (누수 방지)
# ------------------------------------------------------------
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)                   # train만 기준 학습 + 변환 (평균·표준편차 계산)
X_te_s = sc.transform(X_te)                       # test는 변환만 (시험지 미리 안 보기)

In [4]:
# ------------------------------------------------------------
# [목적] 학습 후 ROC-AUC + 클래스별 성적표
# ------------------------------------------------------------
model = LogisticRegression(max_iter=5000).fit(X_tr_s, y_tr)   # 스케일한 값으로 학습
proba = model.predict_proba(X_te_s)[:, 1]         # 양성(1)일 확률
print('ROC-AUC:', round(roc_auc_score(y_te, proba), 3))       # 확률 순위 품질
print(classification_report(y_te, model.predict(X_te_s)))     # 정밀도·재현율·f1

ROC-AUC: 0.996
              precision    recall  f1-score   support

           0       1.00      0.95      0.98        42
           1       0.97      1.00      0.99        72

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [5]:
# ============================================================
# [결과 해석]
#  · ROC-AUC ~ 0.996 -> 거의 완벽하게 양성/악성 '순위'를 가려냄 (1에 근접)
#  · 성적표: 정밀도·재현율 대부분 0.95~1.00, 전체 정확도 ~ 0.98
#  · 스케일링(train만 fit)으로 로지스틱 성능이 안정적으로 높아짐
# ============================================================